# FullStack IA · Infrastructure de données

**Objectif :** Cloud AWS, collecte par API, SQL, ETL et préparation de la certification Cloud Practitioner.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Le cloud et AWS : les bases

Le module Infrastructure de données du FullStack se déroule sur **AWS** et prépare la certification **AWS Cloud Practitioner (CLF-C02)**, avec cinq examens blancs. Cette leçon te donne le vocabulaire pour arriver à l'aise. Fais les manipulations sur ton ordinateur.

## Pourquoi le cloud

Au lieu d'acheter des serveurs, on loue de la puissance de calcul et du stockage à la demande, facturés à l'usage. Trois modèles :
- **IaaS** (*Infrastructure as a Service*) : tu loues des machines virtuelles et tu gères tout le reste (EC2).
- **PaaS** (*Platform as a Service*) : tu déploies ton code, la plateforme gère les serveurs (Elastic Beanstalk, Heroku).
- **SaaS** (*Software as a Service*) : tu utilises un logiciel prêt à l'emploi (Gmail, Notion).

## Les services AWS à connaître pour la formation

| Service | Rôle | Analogie |
| --- | --- | --- |
| **S3** | Stockage d'objets (fichiers) dans des *buckets* | Un disque dur infini en ligne |
| **EC2** | Machines virtuelles | Un ordinateur loué à l'heure |
| **RDS** | Bases de données relationnelles gérées (PostgreSQL…) | Une base SQL sans administration |
| **IAM** | Utilisateurs, rôles et permissions | Le service des badges d'accès |
| **Lambda** | Fonctions exécutées à la demande, sans serveur | Un script qui se réveille sur un événement |
| **SageMaker** | Entraînement et déploiement de modèles de ML | Un atelier de ML géré |
| **Bedrock** | Accès à des modèles de langage via une API | Un catalogue de LLM |

## Les notions qui tombent à l'examen

- **Régions** (par exemple eu-west-3 = Paris) et **zones de disponibilité** (des centres de données isolés dans une région) : on répartit sur plusieurs zones pour résister à une panne.
- Le **modèle de responsabilité partagée** : AWS sécurise *le* cloud (matériel, réseau, bâtiments) ; toi, tu sécurises ce que tu mets *dans* le cloud (données, accès, configuration).
- Le **principe du moindre privilège** : chaque utilisateur ou programme reçoit uniquement les droits dont il a besoin.
- La **tarification** : à l'usage (*on-demand*), réservée (engagement de 1 ou 3 ans, moins cher), ou *spot* (capacité inutilisée, jusqu'à 90 % moins cher mais interruptible).

## Manipuler S3 avec boto3

`boto3` est le SDK Python d'AWS. Ce code se lance sur ton ordinateur, une fois tes identifiants configurés avec `aws configure`.

```python
import boto3

s3 = boto3.client("s3", region_name="eu-west-3")
s3.create_bucket(Bucket="mon-lac-de-donnees-2026",
                 CreateBucketConfiguration={"LocationConstraint": "eu-west-3"})
s3.upload_file("titanic.csv", "mon-lac-de-donnees-2026", "raw/titanic.csv")

for obj in s3.list_objects_v2(Bucket="mon-lac-de-donnees-2026", Prefix="raw/")["Contents"]:
    print(obj["Key"], obj["Size"])

s3.download_file("mon-lac-de-donnees-2026", "raw/titanic.csv", "copie.csv")
```

Un **lac de données** (*data lake*) range les fichiers par zones : `raw/` (brut), `clean/` (nettoyé), `curated/` (prêt à l'analyse).

> **À retenir** : active la facturation par alertes (AWS Budgets) dès la création du compte, n'utilise jamais le compte racine au quotidien, et active l'authentification à deux facteurs. Une clé AWS publiée sur GitHub est exploitée en quelques minutes.

**À faire :**

- [ ] Créer un compte AWS (offre gratuite) et activer l'authentification à deux facteurs sur le compte racine
- [ ] Créer une alerte de budget à 5 € dans AWS Budgets
- [ ] Créer un utilisateur IAM pour tes travaux, sans utiliser le compte racine
- [ ] Installer AWS CLI et lancer aws configure avec cet utilisateur
- [ ] Créer un bucket S3 et y envoyer titanic.csv avec boto3
- [ ] Faire le parcours gratuit « AWS Cloud Practitioner Essentials » sur AWS Skill Builder

---
## SQL : interroger une base de données

**SQL** est le langage des bases de données relationnelles. Chez Jedha, il arrive au module Infrastructure de données (« Bases du SQL », projet Brewly). Bonne nouvelle : la base d'exemple de cette plateforme s'appelle aussi **Brewly**, une chaîne de cafés, et tu l'interroges avec du vrai SQL, exécuté par **SQLite** dans ton navigateur.

Elle contient quatre tables :

| Table | Colonnes |
| --- | --- |
| `clients` | id, prenom, ville, inscrit_le |
| `produits` | id, nom, categorie, prix |
| `commandes` | id, client_id, date, magasin |
| `lignes` | commande_id, produit_id, quantite |

In [ ]:
import pandas as pd
from brewly import connecter

con = connecter()

def sql(requete):
    """Exécute une requête SQL et renvoie un DataFrame."""
    return pd.read_sql_query(requete, con)

sql("SELECT * FROM produits")

## SELECT, WHERE, ORDER BY, LIMIT

L'ordre d'écriture est toujours le même : `SELECT` (quelles colonnes), `FROM` (quelle table), `WHERE` (quelles lignes), `ORDER BY` (dans quel ordre), `LIMIT` (combien).

In [ ]:
sql("""
SELECT nom, prix
FROM produits
WHERE categorie = 'café' AND prix > 3
ORDER BY prix DESC
""")

Quelques opérateurs utiles dans `WHERE` :

In [ ]:
sql("""
SELECT prenom, ville, inscrit_le
FROM clients
WHERE ville IN ('Lyon', 'Lille')
  AND prenom LIKE 'L%'            -- commence par L
  AND inscrit_le BETWEEN '2026-02-01' AND '2026-03-31'
ORDER BY inscrit_le
""")

## Calculer et renommer

In [ ]:
sql("""
SELECT nom,
       prix,
       ROUND(prix * 1.10, 2) AS prix_ttc,
       CASE WHEN prix >= 4 THEN 'premium' ELSE 'standard' END AS gamme
FROM produits
ORDER BY prix_ttc DESC
LIMIT 5
""")

`SELECT DISTINCT` supprime les doublons : `SELECT DISTINCT magasin FROM commandes`.

> **À retenir** : SQL décrit **ce que tu veux**, pas comment le calculer. Écris tes mots-clés en majuscules, une clause par ligne : tes requêtes resteront lisibles quand elles grandiront.

### Exercice 1 · Les thés abordables

Écris dans la variable `requete` une requête SQL qui renvoie le `nom` et le `prix` des produits de la catégorie `thé` coûtant moins de 4,30 €, du moins cher au plus cher. Le code fourni l'exécute dans `res`.

<details><summary>Indice</summary>

WHERE categorie = 'thé' AND prix < 4.3, puis ORDER BY prix.
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()

requete = """

"""
res = pd.read_sql_query(requete, con)
res

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert list(res.columns) == ["nom", "prix"], f"Colonnes : {list(res.columns)}."
assert list(res["nom"]) == ["Thé vert", "Chaï latte"], f"Résultat : {list(res['nom'])}."
print("Bravo, tout est correct.")

### Exercice 2 · Les premières commandes de Bastille

Écris `requete` : les 5 **premières** commandes (par date croissante) passées au magasin `Bastille`, avec les colonnes `id` et `date`. En cas d'égalité de date, trie par `id`.

<details><summary>Indice</summary>

ORDER BY date, id puis LIMIT 5.
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()

requete = """

"""
res = pd.read_sql_query(requete, con)
res

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ref = pd.read_sql_query("SELECT id, date FROM commandes WHERE magasin = 'Bastille' ORDER BY date, id LIMIT 5", con)
assert list(res.columns) == ["id", "date"], f"Colonnes : {list(res.columns)}."
assert res.equals(ref), "Le résultat ne correspond pas : vérifie le filtre, le tri et la limite."
print("Bravo, tout est correct.")

---
## SQL : agréger avec GROUP BY

`GROUP BY` est l'équivalent SQL du `groupby` de pandas : on découpe les lignes en groupes, on applique une fonction d'agrégation, on combine.

*Animation interactive « groupby » : à voir sur la plateforme web.*

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()
sql = lambda q: pd.read_sql_query(q, con)

sql("""
SELECT magasin,
       COUNT(*)            AS nb_commandes,
       MIN(date)           AS premiere,
       MAX(date)           AS derniere
FROM commandes
GROUP BY magasin
ORDER BY nb_commandes DESC
""")

Les fonctions d'agrégation : `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`. `COUNT(DISTINCT client_id)` compte les valeurs différentes.

## Filtrer après agrégation : HAVING

`WHERE` filtre les **lignes avant** le regroupement ; `HAVING` filtre les **groupes après**.

In [ ]:
sql("""
SELECT client_id, COUNT(*) AS nb
FROM commandes
WHERE date >= '2026-04-01'
GROUP BY client_id
HAVING COUNT(*) >= 4
ORDER BY nb DESC
""")

## Grouper par une expression

SQLite stocke ici les dates comme du texte `AAAA-MM-JJ` : `SUBSTR(date, 1, 7)` en extrait le mois. Dans PostgreSQL, tu utiliserais `DATE_TRUNC('month', date)`.

In [ ]:
sql("""
SELECT SUBSTR(date, 1, 7) AS mois, COUNT(*) AS commandes
FROM commandes
GROUP BY mois
ORDER BY mois
""")

> **À retenir** : l'ordre d'exécution logique est FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT. C'est pour cela qu'un alias défini dans SELECT n'est pas toujours utilisable dans WHERE.

### Exercice 1 · Les magasins par mois

Écris `requete` qui compte le nombre de commandes par `magasin` pour le mois de mars 2026 uniquement, dans une colonne `nb`, triée par `nb` décroissant puis par `magasin`.

<details><summary>Indice</summary>

Le filtre sur le mois va dans WHERE (avant le regroupement). Tu peux utiliser BETWEEN ou SUBSTR(date, 1, 7) = '2026-03'.
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()

requete = """

"""
res = pd.read_sql_query(requete, con)
res

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ref = pd.read_sql_query("SELECT magasin, COUNT(*) AS nb FROM commandes WHERE SUBSTR(date,1,7) = '2026-03' GROUP BY magasin ORDER BY nb DESC, magasin", con)
assert list(res.columns) == ["magasin", "nb"], f"Colonnes : {list(res.columns)}."
assert res.equals(ref), "Le résultat ne correspond pas. Vérifie le filtre sur mars et le tri."
print("Bravo, tout est correct.")

### Exercice 2 · Les clients fidèles

Écris `requete` qui renvoie les `client_id` ayant passé **au moins 13 commandes**, avec leur nombre de commandes dans `nb` et le nombre de magasins différents fréquentés dans `nb_magasins`, triés par `nb` décroissant puis par `client_id`.

<details><summary>Indice</summary>

Le seuil porte sur un agrégat : il va dans HAVING. COUNT(DISTINCT magasin) compte les magasins différents.
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()

requete = """

"""
res = pd.read_sql_query(requete, con)
res

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ref = pd.read_sql_query("SELECT client_id, COUNT(*) AS nb, COUNT(DISTINCT magasin) AS nb_magasins FROM commandes GROUP BY client_id HAVING COUNT(*) >= 13 ORDER BY nb DESC, client_id", con)
assert list(res.columns) == ["client_id", "nb", "nb_magasins"], f"Colonnes : {list(res.columns)}."
assert len(ref) > 0 and res.equals(ref), "Le résultat ne correspond pas. Le seuil va dans HAVING, pas dans WHERE."
print("Bravo, tout est correct.")

---
## SQL : jointures et sous-requêtes

Les données sont réparties dans plusieurs tables pour éviter les répétitions (on dit qu'elles sont **normalisées**). Une **jointure** les recolle grâce aux clés : `commandes.client_id` pointe vers `clients.id`.

*Animation interactive « join » : à voir sur la plateforme web.*

## INNER JOIN

Garde uniquement les lignes qui ont une correspondance des deux côtés.

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()
sql = lambda q: pd.read_sql_query(q, con)

sql("""
SELECT c.id, c.date, cl.prenom, cl.ville
FROM commandes AS c
INNER JOIN clients AS cl ON cl.id = c.client_id
ORDER BY c.id
LIMIT 5
""")

## Plusieurs jointures pour calculer un chiffre d'affaires

In [ ]:
sql("""
SELECT p.categorie,
       SUM(l.quantite)                     AS articles,
       ROUND(SUM(l.quantite * p.prix), 2)  AS chiffre_affaires
FROM lignes AS l
JOIN produits AS p ON p.id = l.produit_id
GROUP BY p.categorie
ORDER BY chiffre_affaires DESC
""")

## LEFT JOIN : garder ce qui n'a pas de correspondance

`LEFT JOIN` garde **toutes** les lignes de la table de gauche ; les colonnes de droite valent `NULL` quand il n'y a pas de correspondance. C'est la méthode pour trouver « ceux qui n'ont jamais… ».

In [ ]:
sql("""
SELECT cl.id, cl.prenom, COUNT(c.id) AS nb_commandes
FROM clients AS cl
LEFT JOIN commandes AS c ON c.client_id = cl.id
GROUP BY cl.id
HAVING nb_commandes = 0
""")

## Sous-requêtes et CTE

Une requête peut en utiliser une autre. La forme la plus lisible est la **CTE** (`WITH ... AS`), qui donne un nom à un résultat intermédiaire.

In [ ]:
sql("""
WITH panier AS (
    SELECT l.commande_id, SUM(l.quantite * p.prix) AS montant
    FROM lignes l JOIN produits p ON p.id = l.produit_id
    GROUP BY l.commande_id
)
SELECT c.magasin, ROUND(AVG(panier.montant), 2) AS panier_moyen
FROM panier JOIN commandes c ON c.id = panier.commande_id
GROUP BY c.magasin
ORDER BY panier_moyen DESC
""")

> **À retenir** : INNER JOIN pour l'intersection, LEFT JOIN pour tout garder à gauche. Dès qu'une requête dépasse 15 lignes, découpe-la en CTE.

### Exercice 1 · Chiffre d'affaires par ville

Écris `requete` : le chiffre d'affaires total (`ca`, arrondi à 2 décimales) par **ville des clients**, trié par `ca` décroissant. Il faut joindre les quatre tables.

<details><summary>Indice</summary>

lignes → produits (pour le prix), lignes → commandes → clients (pour la ville).
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()

requete = """

"""
res = pd.read_sql_query(requete, con)
res

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ref = pd.read_sql_query("SELECT cl.ville, ROUND(SUM(l.quantite*p.prix),2) AS ca FROM lignes l JOIN produits p ON p.id=l.produit_id JOIN commandes c ON c.id=l.commande_id JOIN clients cl ON cl.id=c.client_id GROUP BY cl.ville ORDER BY ca DESC", con)
assert list(res.columns) == ["ville", "ca"], f"Colonnes : {list(res.columns)}."
assert res.equals(ref), "Les montants ne correspondent pas. Vérifie tes jointures."
print("Bravo, tout est correct.")

### Exercice 2 · Les clients jamais venus

Écris `requete` qui renvoie l'`id` et le `prenom` des clients qui n'ont **aucune** commande, triés par `id`. Utilise un `LEFT JOIN`.

<details><summary>Indice</summary>

Après un LEFT JOIN, les clients sans commande ont c.id IS NULL.
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()

requete = """

"""
res = pd.read_sql_query(requete, con)
res

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert list(res.columns) == ["id", "prenom"], f"Colonnes : {list(res.columns)}."
assert list(res["id"]) == [56, 57, 58, 59, 60], f"ids obtenus : {list(res['id'])}."
print("Bravo, tout est correct.")

### Exercice 3 · Le produit star de chaque magasin

Écris `requete` (avec une CTE) qui renvoie, pour chaque `magasin`, le `produit` le plus vendu en quantité et cette quantité `qte`, trié par `magasin`. Indice : calcule d'abord les quantités par magasin et produit, puis garde la ligne dont la quantité est le maximum du magasin.

<details><summary>Indice</summary>

WITH ventes AS (...) puis WHERE v.qte = (SELECT MAX(qte) FROM ventes w WHERE w.magasin = v.magasin). Les fonctions de fenêtre (RANK() OVER ...) marchent aussi.
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()

requete = """

"""
res = pd.read_sql_query(requete, con)
res

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ref = pd.read_sql_query("""WITH v AS (SELECT c.magasin, p.nom AS produit, SUM(l.quantite) AS qte FROM lignes l JOIN commandes c ON c.id=l.commande_id JOIN produits p ON p.id=l.produit_id GROUP BY c.magasin, p.nom) SELECT magasin, produit, qte FROM v a WHERE qte = (SELECT MAX(qte) FROM v b WHERE b.magasin=a.magasin) ORDER BY magasin""", con)
assert list(res.columns) == ["magasin", "produit", "qte"], f"Colonnes : {list(res.columns)}."
assert res.sort_values(["magasin", "produit"]).reset_index(drop=True).equals(ref.sort_values(["magasin", "produit"]).reset_index(drop=True)), "Le résultat ne correspond pas."
print("Bravo, tout est correct.")

---
## Projet · Un mini-entrepôt de données Brewly

Objectif : construire, en Python et SQL, un petit **entrepôt de données analytique** à partir de la base transactionnelle Brewly, comme dans les projets « Base de données Brewly vers AWS RDS PostgreSQL » et « Pipeline ETL PeopleFirst ».

Tu vas créer une **table de faits** (une ligne par ligne de commande, avec le montant) et produire les indicateurs qu'un directeur commercial regarde chaque lundi.

**À faire :**

- [ ] Installer PostgreSQL en local (ou avec Docker) et y charger les tables Brewly avec pandas.to_sql
- [ ] Refaire les requêtes du projet dans PostgreSQL (remplace SUBSTR par TO_CHAR(date, 'YYYY-MM'))
- [ ] Bonus : créer une base RDS PostgreSQL sur l'offre gratuite AWS et y charger les données
- [ ] Publier le pipeline sur GitHub avec un README

### Exercice 1 · La table de faits

Avec une seule requête SQL, crée le DataFrame `faits` : une ligne par ligne de commande, avec les colonnes `commande_id`, `date`, `mois` (format `AAAA-MM`), `magasin`, `ville_client`, `produit`, `categorie`, `quantite` et `montant` (quantité × prix).

<details><summary>Indice</summary>

Quatre tables, trois JOIN. SUBSTR(c.date, 1, 7) pour le mois.
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
n = pd.read_sql_query("SELECT COUNT(*) AS n FROM lignes", con)["n"][0]
attendues = ["commande_id", "date", "mois", "magasin", "ville_client", "produit", "categorie", "quantite", "montant"]
assert list(faits.columns) == attendues, f"Colonnes : {list(faits.columns)}."
assert len(faits) == n, f"faits a {len(faits)} lignes, il en faut {n} (une par ligne de commande)."
assert faits["mois"].str.match(r"^\d{4}-\d{2}$").all(), "mois doit être au format AAAA-MM."
print("Bravo, tout est correct.")

### Exercice 2 · Le tableau de bord du lundi

À partir de `faits` (recrée-le), calcule avec pandas : `ca_mensuel` (Series du chiffre d'affaires par mois, arrondi à 2 décimales, triée par mois), `croissance` (variation en % de mars à avril, arrondie à 1 décimale) et `top_categorie` (la catégorie qui rapporte le plus).

<details><summary>Indice</summary>

groupby("mois")["montant"].sum(). La croissance vaut (avril / mars − 1) × 100.
</details>

In [ ]:
import pandas as pd
from brewly import connecter
con = connecter()
faits = pd.read_sql_query("""
SELECT l.commande_id, c.date, SUBSTR(c.date, 1, 7) AS mois, c.magasin,
       cl.ville AS ville_client, p.nom AS produit, p.categorie,
       l.quantite, l.quantite * p.prix AS montant
FROM lignes l
JOIN commandes c ON c.id = l.commande_id
JOIN clients cl ON cl.id = c.client_id
JOIN produits p ON p.id = l.produit_id
""", con)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ref = faits.groupby("mois")["montant"].sum()
assert list(ca_mensuel.index) == sorted(ref.index), "ca_mensuel doit être indexé par mois, dans l'ordre."
assert abs(ca_mensuel.sum() - ref.sum()) < 0.1, "Les montants mensuels ne correspondent pas."
exp = round((ref["2026-04"] / ref["2026-03"] - 1) * 100, 1)
assert abs(croissance - exp) < 0.11, f"croissance vaut {croissance}, j'attendais {exp}."
assert top_categorie == faits.groupby("categorie")["montant"].sum().idxmax(), f"top_categorie vaut {top_categorie!r}."
print("Bravo, tout est correct.")